# Step 4 — Metric View (Measures + Relationships + Joins + Filters)

**Mistplay Genie Workshop**

Step 3 taught Genie the business's *words*. Now we teach it the business's *math*.

A **metric view** is a governed semantic layer in Unity Catalog. It defines, once and for everyone:
- **joins** (relationships between fact and dimension tables),
- a **filter** (e.g. count only `COMPLETED` payments as revenue),
- **measures** (revenue, gross margin %, on-time rate, AOV) with consistent formulas.

We build **two** metric views — because **grain matters**:
- `mv_order_analytics` — order-line grain (revenue, margin %, AOV), with 4 joins + a completed-only filter.
- `mv_shipment_performance` — shipment grain (on-time rate, delivery days).

Putting on-time rate in the order-line view would line-weight it incorrectly — a real modeling lesson.

> **API vs UI:** metric views are UC objects created with SQL; attaching them to the Genie space is a plain REST `add_metric_view` + PATCH. Fully in code.

In [ ]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.metric_view, workshop_lib.questions
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.metric_view, workshop_lib.questions):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import genie_builder as gb
from workshop_lib import metric_view as mv
from workshop_lib import questions
setup_widgets(dbutils)
cfg = WorkshopConfig.from_widgets(dbutils)
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)
print("space:", space_id)

## 1. Create the metric views (SQL DDL)

`mv.build_metric_views` runs `CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML`. Open `workshop_lib/metric_view.py` to read the full YAML — the joins, the `filter: transactions.status = 'COMPLETED'`, and every measure.

In [ ]:
created = mv.build_metric_views(spark, cfg)
for f in created:
    print("created:", f)

## 2. Sanity-check a measure with SQL (`MEASURE(...)`)

Metric views are queried with the `MEASURE()` function. Genie writes this for users automatically; here's one manual check.

In [ ]:
order_mv = cfg.gold(mv.ORDER_MV)
ship_mv = cfg.gold(mv.SHIPMENT_MV)
print("Gross margin % by subcategory (top 5):")
display(spark.sql(f"""
  SELECT `Subcategory`, MEASURE(`Gross Margin Pct`) AS margin_pct, MEASURE(`Total Revenue`) AS revenue
  FROM {order_mv} GROUP BY `Subcategory` ORDER BY revenue DESC LIMIT 5
"""))
print("On-time rate by region:")
display(spark.sql(f"""
  SELECT `Region`, MEASURE(`On Time Rate`) AS on_time_rate, MEASURE(`Shipment Count`) AS shipments
  FROM {ship_mv} GROUP BY `Region` ORDER BY on_time_rate DESC
"""))

## 3. Attach the metric views to the Genie space

We also register the space-level **join specs** so Genie knows the relationships when it needs the base tables. Then round-trip patch.

In [ ]:
builder = client.load_builder(space_id)
builder.add_metric_view(order_mv)
builder.add_metric_view(ship_mv)

# Space-level join hints between the base fact/dim tables.
fo = cfg.gold("fact_orders")
builder.add_join_spec(fo, "fact_orders", cfg.gold("dim_products"), "dim_products",
                      "`fact_orders`.`product_id` = `dim_products`.`product_id`")
builder.add_join_spec(fo, "fact_orders", cfg.gold("dim_geography"), "dim_geography",
                      "`fact_orders`.`geo_id` = `dim_geography`.`geo_id`")
builder.add_join_spec(fo, "fact_orders", cfg.gold("dim_customers"), "dim_customers",
                      "`fact_orders`.`customer_id` = `dim_customers`.`customer_id`")
builder.add_join_spec(fo, "fact_orders", cfg.gold("fact_transactions"), "fact_transactions",
                      "`fact_orders`.`order_id` = `fact_transactions`.`order_id`")

print(builder.summary())
client.patch_space(space_id, title=cfg.genie_space_name,
                   description="Mistplay Genie Workshop — with metric views.",
                   warehouse_id=cfg.warehouse_id, serialized_space=builder.to_serialized())
print("Patched.")

## 4. Verify

In [ ]:
d = client.load_builder(space_id).to_dict()
print("Attached objects:")
for t in d["data_sources"]["tables"]:
    tag = "  [metric view]" if t["identifier"].rsplit(".",1)[-1].startswith("mv_") else ""
    print("  -", t["identifier"], tag)
print("join_specs:", len(d.get("instructions",{}).get("join_specs",[])))

## 5. Try it — the two question sets

In [ ]:
print(questions.render(4))

## Recap & what's next

Genie now computes governed measures consistently. But it still can't handle specific **join paths and exceptions** the metric views don't model — orders that never shipped, refunded-but-shipped cross-checks, refunds as negative revenue.

**→ Step 5** adds curated **examples**: example queries, filters, measures, fields, and joins.